# Lesson 02 - মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক অন্বেষণ

**মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক (MAF)** হল AI এজেন্ট তৈরি করার জন্য একটি একক ফ্রেমওয়ার্ক। এটি চারটি মূল নির্মাণ ব্লকের একটি পরিষ্কার, সংমিশ্রিত আর্কিটেকচার প্রদান করে:

- **ক্লায়েন্ট** – একটি AI মডেল এন্ডপয়েন্টে সংযোগ করে এবং যোগাযোগ পরিচালনা করে
- **এজেন্ট** – একটি ক্লায়েন্টকে নির্দেশনা এবং সরঞ্জাম সংজ্ঞাসহ মোড়ক করে
- **সরঞ্জাম** – এজেন্টের সক্ষমতাগুলি কাস্টম ফাংশন দিয়ে বাড়িয়ে তোলে যা মডেল কল করতে পারে
- **সেশন** – বহু-টর্নের ইন্টারঅ্যাকশনের জন্য কথোপকথনের ইতিহাস রক্ষা করে

এই পাঠে, আমরা একটি **ভ্রমণ বুকিং এজেন্ট** তৈরি করব যা এই ধারণা ব্যবহার করে গন্তব্যের উপলভ্যতা চেক করে।


## সেটআপ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## এজেন্ট ফ্রেমওয়ার্ক আর্কিটেকচার বোঝা

মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক একটি স্তরযুক্ত আর্কিটেকচার অনুসরণ করে:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ক্লায়েন্ট** – একটি `AzureAIProjectAgentProvider` একটি Azure OpenAI ডিপ্লয়মেন্টের সাথে সংযুক্ত হয়। এটি প্রমাণীকরণ, অনুরোধের ফরম্যাটিং, এবং প্রতিক্রিয়া পার্সিং পরিচালনা করে।
2. **এজেন্ট** – ক্লায়েন্ট থেকে `provider.create_agent()` দ্বারা তৈরি, এজেন্ট মডেল অ্যাক্সেস, নির্দেশাবলী (সিস্টেম প্রম্পট) এবং টুলস একত্রিত করে।
3. **টুলস** – পাইথন ফাংশনগুলি যা `@tool` দিয়ে সজ্জিত থাকে, যেগুলি এজেন্ট কার্য সম্পাদন বা তথ্য পুনরুদ্ধারের জন্য আহ্বান করতে পারে।
4. **সেশন** – একটি `AgentSession` অবজেক্ট (যা `agent.create_session()` দ্বারা তৈরি) যা কথোপকথনের ইতিহাস সংরক্ষণ করে, enabling multi-turn dialogue যেখানে এজেন্ট পূর্বের প্রসঙ্গ মনে রাখে।

আসুন প্রতিটি স্তর ধাপে ধাপে তৈরি করি।


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ডেকোরেটর সহ টুল যুক্ত করা

টুলগুলি এজেন্টদেরকে শুধু পাঠ্য উৎপাদনের বাইরে অ্যাকশন নিতে সক্ষম করে। `@tool` ডেকোরেটর একটি সাধারণ পাইথন ফাংশনকে এমন কিছুতে রূপান্তর করে যা এজেন্ট কল করতে পারে।

মূল বিষয়গুলি:
- মডেল প্রতিটি প্যারামিটার বুঝতে পারে সেজন্য `Annotated[type, "description"]` ব্যবহার করুন।
- ডকস্ট্রিং হয় টুলের বর্ণনা যা মডেল দেখে।
- `approval_mode="never_require"` মানে টুলটি স্বয়ংক্রিয়ভাবে চলে ব্যবহারকারীর অনুমতি ছাড়াই।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## টুলস সহ একটি এজেন্ট তৈরি করা

এখন আমরা ক্লায়েন্ট, নির্দেশাবলী, এবং টুলসকে একত্রিত করে একটি এজেন্ট তৈরি করব। `নির্দেশাবলী` সিস্টেম প্রম্পট হিসেবে কাজ করে — এগুলো এজেন্টের ব্যক্তিত্ব এবং আচরণ নির্ধারণ করে।


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## সেশন সহ মাল্টি-টার্ন কথোপকথন

একটি `AgentSession` (যা `agent.create_session()` এর মাধ্যমে তৈরি করা হয়) কথোপকথনের সমস্ত বার্তাগুলির ট্র্যাক রাখে। একই সেশন প্রতিটি `agent.run()` কল-এ প্রদান করলে, এজেন্ট সম্পূর্ণ কথোপকথনের ইতিহাস অ্যাক্সেস করতে পারে এবং পূর্ববর্তী বার্তাগুলির দিকে ফিরে দেখতে পারে।

আমরা `tools=[check_destination_availability]` প্রদান করি যাতে এজেন্ট প্রতিটি টার্নে আমাদের উপলভ্যতা পরীক্ষককে কল করতে পারে।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## সারসংক্ষেপ

এই পাঠে আপনি Microsoft Agent Framework এর চারটি মূল স্তম্ভ পরীক্ষা করেছেন:

| ধারণা | আপনি যা শিখেছেন |
|---------|------------------|
| **ক্লায়েন্ট** | `AzureAIProjectAgentProvider` ক্রেডেনশিয়াল-ভিত্তিক প্রমাণীকরণ সহ Azure OpenAI এর সাথে সংযোগ করে |
| **এজেন্ট** | `provider.create_agent()` একটি মডেল সংযোগকে নির্দেশাবলী এবং একটি নামের সাথে একটি বান্ডেলে বেঁধে দেয় |
| **টুলস** | `@tool` ডেকোরেটর পাইথন ফাংশনসমূহকে প্রকাশ করে যা এজেন্ট কল করতে পারে |
| **সেশন** | `agent.create_session()` একাধিক টার্ন জুড়ে কথোপকথনের ইতিহাস বজায় রাখে |

এই নির্মাণ ব্লকগুলো একসাথে মিলে এমন এজেন্ট তৈরি করে যা স্বাভাবিক কথোপকথন রাখতে পারে, বাহ্যিক ফাংশন কল করতে পারে, এবং প্রাসঙ্গিকতা বজায় রাখতে পারে — পরবর্তী পাঠে আরও উন্নত এজেন্টিক প্যাটার্নের ভিত্তি।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:  
এই ডকুমেন্টটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসম্ভব সঠিকতার চেষ্টা করি, তবে অনুগ্রহ করে জেনে রাখুন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বজাতি ভাষায় প্রামাণিক উৎস হিসাবে গণ্য করা উচিত। গুরুতর বা গুরুত্বপূর্ণ তথ্যের ক্ষেত্রে পেশাদার মানব অনুবাদ গ্রহণ করার পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারে সৃষ্ট কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নয়।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
